# 07 - Route Prediction (Markov Chain)

**CASEFILE: AI-Powered Missing Person Investigation System**  
*Phase 7: Sequential Route Prediction & Corridor Modeling via Markov Chains*

---

### Overview
While location classification predicts an individual's final destination sector, search-and-rescue teams require trajectory corridor intelligence: *which sequence of transit zones did the person likely traverse after disappearing?*

This notebook demonstrates:
1. **First-Order Markov Chain Modeling**: Formulating spatial transitions between functional macro-areas as a stochastic process.
2. **Transition Probability Matrix**: Deriving empirical transfer likelihoods from historical GPS movement sequences.
3. **Topological Network Visualization**: Mapping transition probabilities onto real-world spatial coordinates.
4. **Beam Search Path Decoding**: Implementing a multi-step beam search ($W=3$) to decode the Top-3 most probable route sequences without greedy collapse.
5. **Trajectory Similarity Evaluation**: Benchmarking predicted corridors against historical ground-truth paths using Jaccard and Longest Common Subsequence (LCS) metrics.

> **DISCLAIMER & ETHICAL USAGE STATEMENT**  
> Route predictions represent stochastic trajectory models based on aggregated cohort mobility patterns. They serve solely as heuristic search corridor guidance for search-and-rescue teams and patrol sweeps, not as deterministic evidence of an individual's travel.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Setup visualization styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Directory paths
DATA_DIR = '../data' if os.path.exists('../data') else 'data'
REPORTS_DIR = '../reports' if os.path.exists('../reports') else 'reports'
MODELS_DIR = '../models' if os.path.exists('../models') else 'models'

print("Route prediction environment initialized successfully.")
print(f"Data Directory: {DATA_DIR}")
print(f"Models Directory: {MODELS_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")

## 1. First-Order Markov Chain Formulation

### Mathematical Principles:
- Let $S = \{s_0, s_1, s_2, s_3, s_4\}$ represent the discrete state space of the 5 identified macro-areas.
- Under the **First-Order Markov Property**, the probability of moving to area $S_{t+1}$ depends solely on the current area $S_t$:
  $$P(S_{t+1} = s_j \mid S_t = s_i, S_{t-1} = s_{t-1}, \dots, S_0 = s_0) = P(S_{t+1} = s_j \mid S_t = s_i) = T_{i, j}$$
- The transition probability matrix $T \in \mathbb{R}^{5 \times 5}$ satisfies row-stochasticity:
  $$\sum_{j=0}^{4} T_{i, j} = 1 \quad \forall i \in \{0, 1, 2, 3, 4\}$$
- Transition probabilities are estimated via Maximum Likelihood Estimation (MLE) from consecutive inter-area moves extracted from trajectory histories:
  $$T_{i, j} = \frac{C(s_i \to s_j)}{\sum_{k} C(s_i \to s_k)}$$

We load `models/transition_matrix.pkl` and `data/processed/area_centers.csv`.

In [ ]:
import sys
sys.path.insert(0, '..')

matrix_path = os.path.join(MODELS_DIR, 'transition_matrix.pkl')
area_centers_path = os.path.join(DATA_DIR, 'processed', 'area_centers.csv')

matrix_dict = joblib.load(matrix_path)
matrix = matrix_dict['matrix']
area_to_idx = matrix_dict['area_to_idx']
idx_to_area = matrix_dict['idx_to_area']

area_centers_df = pd.read_csv(area_centers_path)
area_names = dict(zip(area_centers_df['area_id'], area_centers_df['name']))

labels = [f"Area {i}: {area_names.get(i, str(i)).replace('Area near ', '')}" for i in range(len(matrix))]
matrix_df = pd.DataFrame(matrix, index=labels, columns=[f"Area {i}" for i in range(len(matrix))])

print("=" * 65)
print("          EMPIRICAL AREA TRANSITION PROBABILITY MATRIX")
print("=" * 65)
display(matrix_df.round(4))

## 2. Transition Matrix Heatmap Visualization

The transition matrix heatmap highlights key movement corridors:
- **Area 1 $\to$ Area 0**: Very strong transition likelihood ($77.6\%$), representing a primary arterial transit flow between northern hospital districts and central park sectors.
- **Area 0 $\to$ Area 1**: Frequent return commute flow ($46.2\%$).
- **Area 0 $\to$ Area 4 & Area 3**: Secondary branch movements toward southern and eastern transit stops.

In [ ]:
import sys
sys.path.insert(0, '..')

# Display the generated transition matrix report image
tm_img = os.path.join(REPORTS_DIR, 'transition_matrix.png')
if os.path.exists(tm_img):
    print("Loading reports/transition_matrix.png:")
    display(Image(filename=tm_img, width=720))
else:
    print(f"Warning: {tm_img} not found.")

## 3. Spatial Network Transition Graph

The transition graph visualizes the spatial geometry of movements:
- Nodes represent the 5 macro-area centroids placed at their true GPS coordinates.
- Directed arrows represent allowable transitions, with arrow thickness and alpha proportional to transition probability.

In [ ]:
import sys
sys.path.insert(0, '..')

# Display the generated transition network graph report image
tg_img = os.path.join(REPORTS_DIR, 'transition_graph.png')
if os.path.exists(tg_img):
    print("Loading reports/transition_graph.png:")
    display(Image(filename=tg_img, width=820))
else:
    print(f"Warning: {tg_img} not found.")

## 4. Multi-Step Route Decoding via Beam Search

### Why Beam Search?
- **Greedy Search Limitation**: Pure greedy decoding picks $\arg\max_j T_{i, j}$ at every step. This leads directly to degenerate 2-state cycles (e.g., $1 \to 0 \to 1 \to 0$), obscuring alternative plausible routes.
- **Exhaustive Search Limitation**: Evaluating all paths over $N$ steps requires $|S|^N = 5^4 = 625$ evaluations, scaling exponentially.
- **Beam Search Optimization**:
  - Maintains a fixed beam width $W = 3$ of the most promising partial paths ranked by cumulative joint probability:
    $$P(S_1, \dots, S_t) = \prod_{k=1}^t P(S_k \mid S_{k-1})$$
  - At each step $t$, expands all $W$ paths into all possible next states, evaluates their updated joint likelihoods, and retains the top $W$ candidates.
  - Provides diverse, high-confidence tactical route alternatives.

## 5. Predicted Top-3 Routes & Probability Progression

We load `data/processed/predicted_routes.csv`, which contains the Top-3 paths originating from Area 1 across a 4-step horizon.

In [ ]:
import sys
sys.path.insert(0, '..')

routes_path = os.path.join(DATA_DIR, 'processed', 'predicted_routes.csv')
routes_df = pd.read_csv(routes_path)

print("=" * 75)
print("               TOP-3 PREDICTED TRAJECTORY CORRIDORS")
print("=" * 75)
display(routes_df)

In [ ]:
import sys
sys.path.insert(0, '..')

# Visualizing step and cumulative probability decay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2980b9', '#27ae60', '#e67e22']

# 1. Step-by-step probability
for rank in sorted(routes_df['route_rank'].unique()):
    sub = routes_df[routes_df['route_rank'] == rank]
    path_label = " -> ".join([f"A{int(a)}" for a in sub['area_id']])
    ax1.plot(sub['step'], sub['step_probability'], marker='o', linewidth=2.2, 
             color=colors[rank-1], label=f"Route {rank}: {path_label}")

ax1.set_title("Step-by-Step Transition Probabilities", fontweight='bold')
ax1.set_xlabel("Trajectory Hop (Step)")
ax1.set_ylabel("Step Transition Probability P(St | St-1)")
ax1.set_xticks([1, 2, 3, 4])
ax1.set_ylim(0.15, 0.90)
ax1.legend(loc='lower left', frameon=True)
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. Cumulative joint probability
for rank in sorted(routes_df['route_rank'].unique()):
    sub = routes_df[routes_df['route_rank'] == rank]
    final_prob = sub['cumulative_probability'].iloc[-1]
    ax2.plot(sub['step'], sub['cumulative_probability'], marker='s', linewidth=2.2, 
             color=colors[rank-1], label=f"Route {rank} Final Prob: {final_prob:.3f}")

ax2.set_title("Cumulative Joint Path Probability Decay", fontweight='bold')
ax2.set_xlabel("Trajectory Hop (Step)")
ax2.set_ylabel("Cumulative Probability P(S1, ..., St)")
ax2.set_xticks([1, 2, 3, 4])
ax2.set_ylim(0.0, 0.55)
ax2.legend(loc='upper right', frameon=True)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 6. Route Similarity Evaluation vs. Historical Ground Truth

To validate route realism, we evaluate predicted paths against actual historical user trajectory sequences using:
1. **Jaccard Similarity**: Spatial set overlap of visited areas regardless of order:
   $$J(R_{\text{pred}}, R_{\text{hist}}) = \frac{|R_{\text{pred}} \cap R_{\text{hist}}|}{|R_{\text{pred}} \cup R_{\text{hist}}|}
   $$
2. **Longest Common Subsequence (LCS) Match**: Evaluates sequential directionality and order preservation:
   $$\text{Seq Match} = \frac{\text{LCS}(R_{\text{pred}}, R_{\text{hist}})}{\max(|R_{\text{pred}}|, |R_{\text{hist}}|)}$$

In [ ]:
import sys
sys.path.insert(0, '..')

from src.route_prediction import compute_route_similarity

# Representative historical trajectories recorded in the dataset
historical_sequences = [
    [1, 0, 1, 0],
    [1, 0, 4, 2],
    [1, 0, 3, 2],
    [0, 1, 0, 1],
    [2, 3, 0, 1]
]

top_route = routes_df[routes_df['route_rank'] == 1].to_dict('records')
sim_results = compute_route_similarity(top_route, historical_sequences)

eval_df = pd.DataFrame(sim_results)
eval_df['Historical Route Reference'] = [" -> ".join([f"Area {a}" for a in seq]) for seq in historical_sequences]
eval_df['Jaccard Similarity'] = eval_df['jaccard'].apply(lambda x: f"{x:.3f}")
eval_df['Sequence Match (LCS)'] = eval_df['seq_match'].apply(lambda x: f"{x:.3f}")

print("=" * 75)
print("          PREDICTED ROUTE 1 SIMILARITY TO HISTORICAL TRAJECTORIES")
print("=" * 75)
display(eval_df[['Historical Route Reference', 'Jaccard Similarity', 'Sequence Match (LCS)']])

mean_jaccard = eval_df['jaccard'].mean()
mean_lcs = eval_df['seq_match'].mean()
print(f"\nMean Jaccard Similarity:     {mean_jaccard:.4f}")
print(f"Mean Sequence Ordering Match: {mean_lcs:.4f}")

## 7. Search & Rescue Operational Deployment

- **Search Corridor Prioritization**: Rather than organizing unfocused radial searches around the last known position, search resources are deployed along the primary predicted corridor ($1 \to 0 \to 1 \to 0$, $12.8\%$ joint probability) and secondary divergence corridor ($1 \to 0 \to 4 \to 2$).
- **Surveillance Checkpoints**: Transit security and highway camera teams can be immediately instructed to inspect road junctions connecting Area 1 and Area 0.
- **Dynamic Probability Updating**: As search teams clear specific areas without findings, Bayes' rule can condition the transition matrix to redistribute probabilities across remaining branches.